## **Localiser un fichier Wikipédia**

In [1]:
from pathlib import Path

WIKI_DIR = Path("/mnt/c/Users/KSOMS/Favorites/TTA-assan-cyriac-abonouan/fact-checker/data/wikipedia")

wiki_files = sorted(WIKI_DIR.glob("wiki-*.jsonl"))

print(f"Nombre de fichiers Wikipédia : {len(wiki_files)}")
print(f"Premier fichier : {wiki_files[0]}")

Nombre de fichiers Wikipédia : 109
Premier fichier : /mnt/c/Users/KSOMS/Favorites/TTA-assan-cyriac-abonouan/fact-checker/data/wikipedia/wiki-001.jsonl


## **Fonction de lecture**

In [2]:
import json

def load_jsonl(path):
    data = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))

    return data

## **Charger le premier fichier**

In [3]:
wiki_sample = load_jsonl(wiki_files[0])

print(len(wiki_sample))

50000


Le premier fichier contient 50000 articles

In [7]:
wiki_sample[1]

{'id': '1928_in_association_football',
 'text': 'The following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world . ',
 'lines': '0\tThe following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world .\n1\t'}

* **id** : C'est le titre de l'article Wikipédia.
* **text** : C'est le text complet de l'article 
* **line** : Le champs le plus important, il contient le numéro de phrase et le texte (ce que FEVER utilise)

In [8]:
# Afficher les 5 premiers articles
for article in wiki_sample[:5]:
    print("=" * 80)
    print(article["id"])
    print(article["text"][:120])
    print()




1928_in_association_football
The following are the football -LRB- soccer -RRB- events of the year 1928 throughout the world . 

1986_NBA_Finals
The 1986 NBA Finals was the championship round of the 1985 -- 86 NBA season . It pitted the Eastern Conference champion 

1901_Villanova_Wildcats_football_team
The 1901 Villanova Wildcats football team represented the Villanova University during the 1901 college football season .

1992_Northwestern_Wildcats_football_team
The 1992 Northwestern Wildcats team represented Northwestern University during the 1992 NCAA Division I-A football seaso



## **A noter**

Chaque fichier contient 50 000 articles :
* wiki-001.jsonl → 50 000 articles
* wiki-002.jsonl → 50 000 articles

Donc notre prétraitement devra être streaming (lecture ligne par ligne) et non charger tous les fichiers en mémoire. Même si chaque fichier tient en mémoire, l'ensemble de Wikipédia représente plusieurs millions d'articles

De plus, tous les articles ont la même structure (id, text, lines).

## **Nombres de phrase moyen d'un articles wikipedia** 

In [9]:
import numpy as np

sentence_counts = []

for article in wiki_sample:
    count = 0

    for line in article["lines"].split("\n"):
        if not line.strip():
            continue

        parts = line.split("\t", 1)

        if len(parts) < 2:
            continue

        sentence = parts[1].strip()

        if sentence:
            count += 1

    sentence_counts.append(count)

print(f"Nombre moyen de phrases : {np.mean(sentence_counts):.2f}")
print(f"Médiane : {np.median(sentence_counts):.2f}")
print(f"Maximum : {np.max(sentence_counts)}")
print(f"Minimum : {np.min(sentence_counts)}")

Nombre moyen de phrases : 3.41
Médiane : 2.00
Maximum : 663
Minimum : 0


* La plupart des articles sont très courts.
* La médiane est de 2 phrases : cela signifie que plus de la moitié des articles contiennent seulement 2 phrases ou moins.
* Quelques articles sont extrêmement longs (jusqu'à 663 phrases), mais ce sont des exceptions.

Autrement dit, la distribution est très asymétrique.